# 05- Model optimization

## Objectif
Optimiser SVC avec RandomizedSearchCV

In [1]:
import pandas as pd
import joblib
import json
from pathlib import Path

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

from sklearn.svm import SVC

df = pd.read_csv("../01_Data/processed/churn_features.csv")

if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

X = df.drop(columns=["Churn"])
y = df["Churn"]

categorical_cols = X.select_dtypes(include=["object", "category","bool"]).columns.tolist()
numeric_cols = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", SVC(class_weight="balanced", probability=True, random_state=42)) 
])

param_distributions = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__gamma": ["scale", "auto", 0.001, 0.01, 0.1, 1],
    "model__kernel": ["rbf", "poly", "sigmoid"],
}

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=4,        
    verbose=1,
    error_score="raise",
)

search.fit(X_train, y_train)

print("Meilleurs paramètres :", search.best_params_)
print("Meilleur F1-score CV :", search.best_score_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Meilleurs paramètres : {'model__kernel': 'poly', 'model__gamma': 'auto', 'model__C': 0.1}
Meilleur F1-score CV : 0.6341471134762978


## Interprétation
Cette optimisation permet de démontrer un travail d'amélioration de la performance du modèle.


In [ ]:
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred)),
    "recall": float(recall_score(y_test, y_pred)),
    "f1_score": float(f1_score(y_test, y_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_proba)),
}

print(metrics)
print(classification_report(y_test, y_pred))

In [ ]:
model_dir = Path("../04_Models")
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, model_dir / "best_model.pkl")

metadata = {
    "optimized_model": "SVC",
    "optimization_method": "RandomizedSearchCV",
    "scoring": "f1",
    "best_params": search.best_params_,
    "best_cv_f1": float(search.best_score_),
    "test_metrics": metrics,
    "business_justification": (
    "SVC a été optimisé car il a obtenu le meilleur Recall et le meilleur F1-score "
    "lors de la comparaison initiale des modèles. Ces métriques sont prioritaires "
    "dans un projet de prédiction du churn."
    )
}

with open(model_dir / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
    
print("Modèle SVC optimisé sauvegardé.")

Modèle SVC optimisé sauvegardé.


Exception ignored in: <function ResourceTracker.__del__ at 0x7fe67f16fc40>
Traceback (most recent call last):
  File "/home/jes/miniconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/jes/miniconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/jes/miniconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7fe92f56bc40>
Traceback (most recent call last):
  File "/home/jes/miniconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/jes/miniconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/jes/miniconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7